# Multi-Source RAG with Re-ranking
Combining multiple source formats, sub-query generation, and Cross-Encoder re-ranking.

In [1]:
import os
import csv
from dotenv import load_dotenv
from groq import Groq
import chromadb
from chromadb.utils import embedding_functions
from sentence_transformers import CrossEncoder

load_dotenv()
groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Initialize ChromaDB in a new directory so we don't mess up our previous project
chroma_client = chromadb.PersistentClient(path="./chroma_db_multi")
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = chroma_client.get_or_create_collection(name="multi_source_collection", embedding_function=emb_fn)

# Initialize Re-ranker (downloads automatically on first run)
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Setup complete!")


c:\Users\bikra\OneDrive\Desktop\internship\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2206.62it/s]


Setup complete!


### Step 1: Ingest Multi-Source Data
We will create and ingest a CSV file (table data) and a TXT file (article), tagging them with `source_type` metadata.

In [9]:
os.makedirs('multi_docs', exist_ok=True)

# 1. Create a dummy CSV file
csv_path = "multi_docs/sales_data.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Quarter", "Revenue", "Top_Product"])
    writer.writerow(["Q1", "$500,000", "AI Router Widget"])
    writer.writerow(["Q2", "$750,000", "RAG Pipeline Pro"])

# 2. Create a dummy TXT file
txt_path = "multi_docs/company_update.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    f.write("In Q1, we focused entirely on building our initial AI Router Widget. Sales were steady, but the real breakthrough happened in Q2. In Q2, we launched the RAG Pipeline Pro, which caused our revenue to skyrocket. Customers praised its agentic capabilities.")

# Chunking & Ingestion
all_chunks = []
all_metadata = []
all_ids = []

# Read CSV
with open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for idx, row in enumerate(reader):
        # Convert row to string chunk
        chunk = f"CSV Data -> Quarter: {row['Quarter']}, Revenue: {row['Revenue']}, Top Product: {row['Top_Product']}"
        all_chunks.append(chunk)
        all_metadata.append({"source_type": "csv", "filename": "sales_data.csv"})
        all_ids.append(f"csv_chunk_{idx}")

# Read TXT (using simple sentence splitting for this small doc)
with open(txt_path, "r", encoding="utf-8") as f:
    text = f.read()
    sentences = text.split(". ")
    for idx, sentence in enumerate(sentences):
        if sentence.strip():
            all_chunks.append(sentence.strip())
            all_metadata.append({"source_type": "txt", "filename": "company_update.txt"})
            all_ids.append(f"txt_chunk_{idx}")

collection.upsert(
    documents=all_chunks,
    metadatas=all_metadata,
    ids=all_ids
)
print("Ingested CSV and TXT data into ChromaDB!")


Ingested CSV and TXT data into ChromaDB!


### Step 2: Query Rewriting (Sub-queries)
Break complex questions into 2 sub-queries to improve search coverage.

In [10]:
def generate_subqueries(question):
    print(f"\n[LOG] Original Question: {question}")
    prompt = f"""You are an AI assistant that breaks down complex user queries.
    If the query is multi-part or comparing two things, break it into exactly 2 simple sub-queries.
    If the query is simple, just return the original query as 1 sub-query.
    Format your output strictly as a Python list of strings.
    Example: ["query 1", "query 2"]
    
    Question: {question}"""
    
    response = groq_client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="openai/gpt-oss-120b", 
        temperature=0.0
    )
    
    try:
        # Clean the output string to parse the list securely
        output = response.choices[0].message.content.strip()
        if "[" in output and "]" in output:
            output = output[output.find("["):output.rfind("]")+1]
        subqueries = eval(output)
        if not isinstance(subqueries, list):
            subqueries = [question]
    except:
        subqueries = [question]
        
    print(f"[LOG] Generated Subqueries: {subqueries}")
    return subqueries


### Step 3: Retrieval & Cross-Encoder Re-Ranking
Retrieve for all sub-queries, combine into a big list, and re-rank with MS-MARCO.

In [11]:
def retrieve_and_rerank(subqueries, original_query):
    all_retrieved_chunks = set() # use set to avoid duplicates
    
    # 1. Broad Retrieval (Get top 5 for each sub-query, combining to a large candidate set)
    for sq in subqueries:
        results = collection.query(
            query_texts=[sq],
            n_results=5
        )
        # We check if anything was returned to avoid errors
        if len(results['documents']) > 0:
            for chunk in results['documents'][0]:
                all_retrieved_chunks.add(chunk)
            
    candidates = list(all_retrieved_chunks)
    print(f"[LOG] Broad retrieval found {len(candidates)} unique candidates across sub-queries.")
    
    if not candidates:
        return []

    # 2. Re-Ranking using Cross-Encoder
    # The cross-encoder takes pairs: [(original_query, chunk1), (original_query, chunk2)]
    pairs = [[original_query, doc] for doc in candidates]
    scores = reranker.predict(pairs)
    
    # Sort candidates by their score (highest first)
    scored_candidates = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    
    print("\n[LOG] Re-ranking Scores:")
    for doc, score in scored_candidates:
        print(f"  -> Score {score:.2f} | {doc[:50]}...")
        
    # 3. Select Top 3 absolute best chunks
    top_3 = [doc for doc, score in scored_candidates[:3]]
    return top_3


### Step 4: Final Generation
Combining the exact pieces together.

In [12]:
def multi_source_rag(question):
    print(f"\n{'='*60}")
    
    # 1. Break down the query
    subqueries = generate_subqueries(question)
    
    # 2. Retrieve broadly and re-rank strictly
    top_chunks = retrieve_and_rerank(subqueries, question)
    
    if not top_chunks:
        print("No relevant information found.")
        return
        
    # 3. Final generation
    context = "\n\n---\n\n".join(top_chunks)
    
    prompt = f"""Answer the question using ONLY the provided context.
    
    Context:
    {context}
    
    Question: {question}"""
    
    response = groq_client.chat.completions.create(
        messages=[
            {"role": "system", "content": "You are an analytical AI assistant."},
            {"role": "user", "content": prompt}
        ],
        model="openai/gpt-oss-120b", 
        temperature=0.2
    )
    answer = response.choices[0].message.content
    
    print(f"\n--- FINAL ANSWER ---\n{answer}")
    return answer


### Step 5: Test Multi-Part Queries

In [13]:
# This will test breaking the query down and pulling from BOTH the CSV and TXT files!
multi_source_rag("Compare what the CSV sales data says about Q2 revenue with what the company update text says about Q2's breakthrough.")




[LOG] Original Question: Compare what the CSV sales data says about Q2 revenue with what the company update text says about Q2's breakthrough.
[LOG] Generated Subqueries: ['What does the CSV sales data say about Q2 revenue?', "What does the company update text say about Q2's breakthrough?"]
[LOG] Broad retrieval found 6 unique candidates across sub-queries.

[LOG] Re-ranking Scores:
  -> Score 1.59 | CSV Data -> Quarter: Q2, Revenue: $750,000, Top Pr...
  -> Score 0.30 | CSV Data -> Quarter: Q1, Revenue: $500,000, Top Pr...
  -> Score 0.18 | Sales were steady, but the real breakthrough happe...
  -> Score -3.14 | In Q2, we launched the RAG Pipeline Pro, which cau...
  -> Score -7.69 | In Q1, we focused entirely on building our initial...
  -> Score -8.78 | Customers praised its agentic capabilities....

--- FINAL ANSWER ---
The CSV data provides a concrete figure for Q2 – revenue reached **$750,000**, showing a measurable increase from Q1.  
The company update, however, speaks only i

'The CSV data provides a concrete figure for Q2 – revenue reached **$750,000**, showing a measurable increase from Q1.  \nThe company update, however, speaks only in qualitative terms: it notes that **sales were steady** but that a **“real breakthrough” occurred in Q2**. The update does not give a specific revenue number or directly link the breakthrough to the $750,000 figure; it simply highlights that something notable happened in Q2 beyond the steady sales trend.'